In [12]:
import pandas as pd
from pathlib import Path
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials
from gspread_formatting import *

In [31]:
DIR_PATH = Path("/home/cognify/Downloads/SEC_Funds/Cliffwater_Enhanced_Lending_Fund")
CONSOLIDATED_CSV_PATH = Path(DIR_PATH, "CELF_output/ea0205199-01_ncsr.csv")
ANALYSIS_PATH = Path(DIR_PATH, "analysis")
ANALYSIS_PATH.mkdir(exist_ok=True)
DIR_PATH.exists(), CONSOLIDATED_CSV_PATH.exists(), ANALYSIS_PATH.exists()

(True, True, True)

In [32]:
df_investment = pd.read_csv(CONSOLIDATED_CSV_PATH)
df_investment.info()

<class 'pandas.DataFrame'>
RangeIndex: 405 entries, 0 to 404
Data columns (total 17 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   portfolio_company       405 non-null    str    
 1   investment_type         405 non-null    str    
 2   interest_rate           184 non-null    str    
 3   reference_rate          123 non-null    str    
 4   basis_points_spread     123 non-null    float64
 5   maturity_date           170 non-null    str    
 6   currency                300 non-null    str    
 7   principal_amount        405 non-null    int64  
 8   cost                    396 non-null    str    
 9   fair_value              299 non-null    str    
 10  footnotes               0 non-null      float64
 11  first_acquisition_date  102 non-null    str    
 12  asset_class             405 non-null    str    
 13  industry                405 non-null    str    
 14  part                    405 non-null    str    
 15  

In [33]:
# convert cost 
for col in ["cost", "fair_value"]:
    df_investment[col] = (
        df_investment[col]
        .astype(str)
        .str.replace(",", "", regex=False)
        .str.replace(r"^\((.*)\)$", r"-\1", regex=True)
        .replace("nan", pd.NA)
    )

    df_investment[col] = pd.to_numeric(df_investment[col], errors="coerce").astype("Int64")

df_investment.drop(columns=["part", "table_index", "row_index", "footnotes"], inplace=True)
df_investment

,portfolio_company,investment_type,interest_rate,reference_rate,basis_points_spread,maturity_date,currency,principal_amount,cost,fair_value,first_acquisition_date,asset_class,industry
0,AG Asset Based Credit Fund L.P.,Investment Partnership,NaN,NaN,NaN,NaN,USD,0,82500000,88644558,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
1,"AG Essential Housing Fund II Holdings (DE), L.P.",Investment Partnership,NaN,NaN,NaN,NaN,USD,0,12075000,13280314,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
2,"Ares Commercial Finance, LP",Investment Partnership,NaN,NaN,NaN,NaN,USD,0,28535713,34129618,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
3,"Ares Pathfinder Fund II (Offshore), LP",Investment Partnership,NaN,NaN,NaN,NaN,USD,0,1831168,1907110,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
4,Ares Priority Loan Co-Invest LP,Investment Partnership,NaN,NaN,NaN,NaN,USD,0,28625000,29357914,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,"VPC COV, L.P.",Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,1000000,1000000,<NA>,4/19/2023,Short-Term Investments — 6.1%,Technology — 0.0%
401,"VPC Legal Finance Fund, L.P.",Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,86429056,86429056,<NA>,9/29/2022,Short-Term Investments — 6.1%,Technology — 0.0%
402,Waccamaw River LLC,Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,12498740,12498740,<NA>,8/4/2021,Short-Term Investments — 6.1%,Technology — 0.0%
403,"WhiteHawk Evergreen Fund, LP",Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,50000000,50000000,<NA>,1/31/2024,Short-Term Investments — 6.1%,Technology — 0.0%


In [34]:
# df_investment["profit"] = df_investment["fair_value"] - df_investment["cost"]
# df_investment["is_profit"] = df_investment["profit"] > 0
# df_investment["is_loss"] = df_investment["profit"] < 0
# df_investment["is_breakeven"] = df_investment["profit"] == 0
# df_investment

In [35]:
# df_summary = pd.DataFrame({
#     "a": [
#         "Total Investment Count",
#         "Total Prinicipal Amount", 
#         "Total Cost", 
#         "Total FV", 
#         "P&L"
#     ],
#     "b": [
#         len(df_investment),
#         int(df_investment["principal_amount"].sum()),
#         int(df_investment["cost"].sum()),
#         int(df_investment["fair_value"].sum()),
#         int(df_investment["fair_value"].sum())-int(df_investment["cost"].sum()),
#     ]
# })

# df_summary

In [36]:
# # Company-Level Analysis
# df_company_level_analysis = (
#     df_investment.groupby("portfolio_company")
#       .agg(
#           total_investments=("portfolio_company", "count"),

#           # counts
#           profitable_count=("is_profit", "sum"),
#           loss_count=("is_loss", "sum"),
#           breakeven_count=("is_breakeven", "sum"),

#           # financials
#           total_cost=("cost", "sum"),
#           total_fair_value=("fair_value", "sum"),
#           total_profit=("profit", "sum"),

#           # averages
#           avg_profit=("profit", "mean"),

#           # risk
#           max_profit=("profit", "max"),
#           max_loss=("profit", "min")
#       )
# )

# df_company_level_analysis.reset_index(names="Portfolio Company", inplace=True)
# df_company_level_analysis["avg_profit"] = round(df_company_level_analysis["avg_profit"], 2)

# # win, breakeven, and loss rate
# df_company_level_analysis["win_rate"] = round(
#     df_company_level_analysis["profitable_count"] /
#     df_company_level_analysis["total_investments"]
# , 2)

# df_company_level_analysis["breakeven_rate"] = round(
#     df_company_level_analysis["breakeven_count"] /
#     df_company_level_analysis["total_investments"]
# , 2)

# df_company_level_analysis["loss_rate"] = round(
#     df_company_level_analysis["loss_count"] /
#     df_company_level_analysis["total_investments"]
# , 2)

# # sort by total no investments
# df_company_level_analysis = df_company_level_analysis.sort_values(
#     by="total_fair_value", ascending=False
# )


# column_mapping = {
#     "total_investments": "Total Investment Count",
#     "total_cost": "Total Investment Cost",
#     "total_fair_value": "Total Investment FV",
#     "total_profit": "Total Investment P&L",
#     "avg_profit": "Avg Profit/Investment",
#     "max_profit": "Max Profit Investment",
#     "max_loss": "Max Loss Investment",
#     "profitable_count": "Profitable Investment Count",
#     "win_rate": "Profitable %",
#     "breakeven_count": "Breakeven Investment Count",
#     "breakeven_rate": "Breakeven %",
#     "loss_count": "Loss Investment Count",
#     "loss_rate": "Loss %"
# }
# desired_order = [
#     "Portfolio Company",
#     "Total Investment Count",
#     "Total Investment Cost",
#     "Total Investment FV",
#     "Total Investment P&L",
#     "Avg Profit/Investment",
#     "Max Profit Investment",
#     "Max Loss Investment",
#     "Profitable Investment Count",
#     "Profitable %",
#     "Breakeven Investment Count",
#     "Breakeven %",
#     "Loss Investment Count",
#     "Loss %"
# ]

# df_company_level_analysis = df_company_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

# display(df_company_level_analysis)

In [37]:
# # Industry-Level Analysis
# df_industry_level_analysis = (
#     df_investment.groupby("industry")
#       .agg(
#           total_investments=("portfolio_company", "count"),

#           # counts
#           profitable_count=("is_profit", "sum"),
#           loss_count=("is_loss", "sum"),
#           breakeven_count=("is_breakeven", "sum"),

#           # financials
#           total_cost=("cost", "sum"),
#           total_fair_value=("fair_value", "sum"),
#           total_profit=("profit", "sum"),

#           # averages
#           avg_profit=("profit", "mean"),

#           # risk
#           max_profit=("profit", "max"),
#           max_loss=("profit", "min")
#       )
# )

# df_industry_level_analysis.reset_index(names="Industry", inplace=True)
# df_industry_level_analysis["avg_profit"] = round(df_industry_level_analysis["avg_profit"], 2)

# # win, breakeven, and loss rate
# df_industry_level_analysis["win_rate"] = round(
#     df_industry_level_analysis["profitable_count"] /
#     df_industry_level_analysis["total_investments"]
# , 2)

# df_industry_level_analysis["breakeven_rate"] = round(
#     df_industry_level_analysis["breakeven_count"] /
#     df_industry_level_analysis["total_investments"]
# , 2)

# df_industry_level_analysis["loss_rate"] = round(
#     df_industry_level_analysis["loss_count"] /
#     df_industry_level_analysis["total_investments"]
# , 2)

# # sort by total no investments
# df_industry_level_analysis = df_industry_level_analysis.sort_values(
#     by="total_fair_value", ascending=False
# )


# column_mapping = {
#     "total_investments": "Total Investment Count",
#     "total_cost": "Total Investment Cost",
#     "total_fair_value": "Total Investment FV",
#     "total_profit": "Total Investment P&L",
#     "avg_profit": "Avg Profit/Investment",
#     "max_profit": "Max Profit Investment",
#     "max_loss": "Max Loss Investment",
#     "profitable_count": "Profitable Investment Count",
#     "win_rate": "Profitable %",
#     "breakeven_count": "Breakeven Investment Count",
#     "breakeven_rate": "Breakeven %",
#     "loss_count": "Loss Investment Count",
#     "loss_rate": "Loss %"
# }
# desired_order = [
#     "Industry",
#     "Total Investment Count",
#     "Total Investment Cost",
#     "Total Investment FV",
#     "Total Investment P&L",
#     "Avg Profit/Investment",
#     "Max Profit Investment",
#     "Max Loss Investment",
#     "Profitable Investment Count",
#     "Profitable %",
#     "Breakeven Investment Count",
#     "Breakeven %",
#     "Loss Investment Count",
#     "Loss %"
# ]

# df_industry_level_analysis = df_industry_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

# display(df_industry_level_analysis)

In [38]:
# industry_list = list(df_industry_level_analysis["Industry"])

# arrs = [v.split(" ") for v in industry_list]

# perc = [v[-1] for v in arrs]
# last = [v.pop() for v in arrs]

# names = [" ".join(v) for v in arrs]

# df_industry_level_analysis = df_industry_level_analysis.drop(columns=["Industry"])

# df_industry_level_analysis.insert(0, "Industry", names)
# df_industry_level_analysis.insert(1, "Percentage", perc)
# display(df_industry_level_analysis)

In [39]:
# # Investment Type Level Analysis
# df_investment_type_level_analysis = (
#     df_investment.groupby("investment_type")
#       .agg(
#           total_investments=("portfolio_company", "count"),

#           # counts
#           profitable_count=("is_profit", "sum"),
#           loss_count=("is_loss", "sum"),
#           breakeven_count=("is_breakeven", "sum"),

#           # financials
#           total_cost=("cost", "sum"),
#           total_fair_value=("fair_value", "sum"),
#           total_profit=("profit", "sum"),

#           # averages
#           avg_profit=("profit", "mean"),

#           # risk
#           max_profit=("profit", "max"),
#           max_loss=("profit", "min")
#       )
# )

# df_investment_type_level_analysis.reset_index(names="Investment Type", inplace=True)
# df_investment_type_level_analysis["avg_profit"] = round(df_investment_type_level_analysis["avg_profit"], 2)

# # win, breakeven, and loss rate
# df_investment_type_level_analysis["win_rate"] = round(
#     df_investment_type_level_analysis["profitable_count"] /
#     df_investment_type_level_analysis["total_investments"]
# , 2)

# df_investment_type_level_analysis["breakeven_rate"] = round(
#     df_investment_type_level_analysis["breakeven_count"] /
#     df_investment_type_level_analysis["total_investments"]
# , 2)

# df_investment_type_level_analysis["loss_rate"] = round(
#     df_investment_type_level_analysis["loss_count"] /
#     df_investment_type_level_analysis["total_investments"]
# , 2)

# # sort by total no investments
# df_investment_type_level_analysis = df_investment_type_level_analysis.sort_values(
#     by="total_fair_value", ascending=False
# )


# column_mapping = {
#     "total_investments": "Total Investment Count",
#     "total_cost": "Total Investment Cost",
#     "total_fair_value": "Total Investment FV",
#     "total_profit": "Total Investment P&L",
#     "avg_profit": "Avg Profit/Investment",
#     "max_profit": "Max Profit Investment",
#     "max_loss": "Max Loss Investment",
#     "profitable_count": "Profitable Investment Count",
#     "win_rate": "Profitable %",
#     "breakeven_count": "Breakeven Investment Count",
#     "breakeven_rate": "Breakeven %",
#     "loss_count": "Loss Investment Count",
#     "loss_rate": "Loss %"
# }
# desired_order = [
#     "Investment Type",
#     "Total Investment Count",
#     "Total Investment Cost",
#     "Total Investment FV",
#     "Total Investment P&L",
#     "Avg Profit/Investment",
#     "Max Profit Investment",
#     "Max Loss Investment",
#     "Profitable Investment Count",
#     "Profitable %",
#     "Breakeven Investment Count",
#     "Breakeven %",
#     "Loss Investment Count",
#     "Loss %"
# ]

# df_investment_type_level_analysis = df_investment_type_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

In [40]:
# Fromatting df_investments
column_mapping = {
    "portfolio_company": "Portfolio Company",
    "investment_type": "Investment Type",
    "interest_rate": "Interest Rate",
    "reference_rate": "Reference Rate",
    "basis_points_spread": "Basis Points Spread",
    "maturity_date": "Maturity Date",
    "currency": "Currency",
    "principal_amount": "Principal Amount",
    "cost": "Cost",
    "fair_value": "Fair Value",
    "first_acquisition_date": "First Acquisition Date",
    "asset_class": "Asset Class",
    "industry": "Industry",
    "profit": "Profit"
}

desired_order = [
    "Portfolio Company",
    "Investment Type",
    "Interest Rate",
    "Reference Rate",
    "Basis Points Spread",
    "Maturity Date",
    "Currency",
    "Principal Amount",
    "Cost",
    "Fair Value",
    "Profit",
    "First Acquisition Date",
    "Asset Class",
    "Industry"
]
df_investment_formatted = df_investment.rename(columns=column_mapping).reindex(columns=desired_order)
df_investment_formatted

,Portfolio Company,Investment Type,Interest Rate,Reference Rate,Basis Points Spread,Maturity Date,Currency,Principal Amount,Cost,Fair Value,Profit,First Acquisition Date,Asset Class,Industry
0,AG Asset Based Credit Fund L.P.,Investment Partnership,NaN,NaN,NaN,NaN,USD,0,82500000,88644558,NaN,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
1,"AG Essential Housing Fund II Holdings (DE), L.P.",Investment Partnership,NaN,NaN,NaN,NaN,USD,0,12075000,13280314,NaN,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
2,"Ares Commercial Finance, LP",Investment Partnership,NaN,NaN,NaN,NaN,USD,0,28535713,34129618,NaN,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
3,"Ares Pathfinder Fund II (Offshore), LP",Investment Partnership,NaN,NaN,NaN,NaN,USD,0,1831168,1907110,NaN,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
4,Ares Priority Loan Co-Invest LP,Investment Partnership,NaN,NaN,NaN,NaN,USD,0,28625000,29357914,NaN,NaN,Private Investment Vehicles — 78.4%,Investment Partnerships — 66.6%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
400,"VPC COV, L.P.",Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,1000000,1000000,<NA>,NaN,4/19/2023,Short-Term Investments — 6.1%,Technology — 0.0%
401,"VPC Legal Finance Fund, L.P.",Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,86429056,86429056,<NA>,NaN,9/29/2022,Short-Term Investments — 6.1%,Technology — 0.0%
402,Waccamaw River LLC,Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,12498740,12498740,<NA>,NaN,8/4/2021,Short-Term Investments — 6.1%,Technology — 0.0%
403,"WhiteHawk Evergreen Fund, LP",Private Investment Vehicle,NaN,NaN,NaN,NaN,NaN,50000000,50000000,<NA>,NaN,1/31/2024,Short-Term Investments — 6.1%,Technology — 0.0%


In [41]:
# # asset-level Analysis
# df_asset_level_analysis = (
#     df_investment.groupby("asset_class")
#       .agg(
#           total_investments=("portfolio_company", "count"),

#           # counts
#           profitable_count=("is_profit", "sum"),
#           loss_count=("is_loss", "sum"),
#           breakeven_count=("is_breakeven", "sum"),

#           # financials
#           total_cost=("cost", "sum"),
#           total_fair_value=("fair_value", "sum"),
#           total_profit=("profit", "sum"),

#           # risk
#           max_profit=("profit", "max"),
#           max_loss=("profit", "min")
#       )
# )

# df_asset_level_analysis.reset_index(names="Industry", inplace=True)

# # win, breakeven, and loss rate
# df_asset_level_analysis["win_rate"] = round(
#     df_asset_level_analysis["profitable_count"] /
#     df_asset_level_analysis["total_investments"]
# , 2)

# df_asset_level_analysis["breakeven_rate"] = round(
#     df_asset_level_analysis["breakeven_count"] /
#     df_asset_level_analysis["total_investments"]
# , 2)

# df_asset_level_analysis["loss_rate"] = round(
#     df_asset_level_analysis["loss_count"] /
#     df_asset_level_analysis["total_investments"]
# , 2)

# # FIX 1: Sort df_asset_level_analysis using its own column
# df_asset_level_analysis = df_asset_level_analysis.sort_values(
#     by="total_fair_value", ascending=False
# )

# column_mapping = {
#     "total_investments": "Total Investment Count",
#     "total_cost": "Total Investment Cost",
#     "total_fair_value": "Total Investment FV",
#     "total_profit": "Total Investment P&L",
#     "avg_profit": "Avg Profit/Investment",
#     "max_profit": "Max Profit Investment",
#     "max_loss": "Max Loss Investment",
#     "profitable_count": "Profitable Investment Count",
#     "win_rate": "Profitable %",
#     "breakeven_count": "Breakeven Investment Count",
#     "breakeven_rate": "Breakeven %",
#     "loss_count": "Loss Investment Count",
#     "loss_rate": "Loss %"
# }

# desired_order = [
#     "Industry",
#     "Total Investment Count",
#     "Total Investment Cost",
#     "Total Investment FV",
#     "Total Investment P&L",
#     "Avg Profit/Investment",
#     "Max Profit Investment",
#     "Max Loss Investment",
#     "Profitable Investment Count",
#     "Profitable %",
#     "Breakeven Investment Count",
#     "Breakeven %",
#     "Loss Investment Count",
#     "Loss %"
# ]

# df_asset_level_analysis = df_asset_level_analysis.rename(columns=column_mapping).reindex(columns=desired_order)

# display(df_asset_level_analysis)


In [42]:
# industry_list = list(df_asset_level_analysis["Industry"])

# arrs = [v.split(" ") for v in industry_list]

# perc = [v[-1] for v in arrs]
# last = [v.pop() for v in arrs]

# names = [" ".join(v) for v in arrs]

# df_asset_level_analysis = df_asset_level_analysis.drop(columns=["Industry"])

# df_asset_level_analysis.insert(0, "Industry", names)
# df_asset_level_analysis.insert(1, "Percentage", perc)
# display(df_asset_level_analysis)

### Dumping to GS

In [43]:
SCOPES = [
    "https://www.googleapis.com/auth/spreadsheets",
    "https://www.googleapis.com/auth/drive"
]

creds = Credentials.from_service_account_file(
    "service_account.json",
    scopes=SCOPES
)

client = gspread.authorize(creds)

In [44]:
df_map = {
    # "Summary": df_summary,
    "All Investments": df_investment_formatted,
    # "Industry Level Analysis": df_industry_level_analysis,
    # "Investment Type Analysis": df_investment_type_level_analysis,
    # "Asset Level Analysis": df_asset_level_analysis,
    # "Company Level Analysis": df_company_level_analysis
}

In [46]:
spreadsheet = client.open("CELF-ea0205199-01_ncsr")

In [47]:
for tab_name, df_analysis in df_map.items():
    try:
        worksheet = spreadsheet.worksheet(tab_name)
        print(tab_name)
        worksheet.clear()
    except gspread.WorksheetNotFound:
        worksheet = spreadsheet.add_worksheet(
            title=tab_name,
            rows=max(len(df_analysis) + 10, 100),
            cols=max(len(df_analysis.columns) + 10, 20)
        )

    # display(df_analysis)
    set_with_dataframe(worksheet, df_analysis)

    # Sheet formatting
    header_fmt = CellFormat(
        backgroundColor=Color(0.85, 0.90, 1.0),
        textFormat=TextFormat(bold=True)
    )

    format_cell_range(
        worksheet,
        "1:1",
        header_fmt
    )

    worksheet.freeze(rows=1)
    worksheet.columns_auto_resize(0, len(df_analysis.columns))

## Finding Unique values 

In [18]:
cols = ['investment_type', 'interest_rate','reference_rate','basis_points_spread', 'asset_class', 'industry']

In [22]:
for i  in df_investment['interest_rate'].unique(): print(i)

nan
12.88%
15.67%, 4.00% PIK
9.05%
12.30%
13.54%
13.09%
12.81%
12.94%
12.90%
12.66%
1.00%
0.50%
8.70%
8.45%
8.42%
9.92%
11.20%
11.00%
10.16%
11.53%
13.00%, 8.00% PIK
14.50%
15.25%
14.75%
10.11%
12.67%
27.68%
12.35%
12.17%
11.77%
9.89%
11.00%, 5.00% PIK
8.46%
10.50%
8.49%
15.63% PIK
11.17%
11.15%
11.73%
10.77%
9.26%
10.69%
11.79%
12.63%
10.93%
14.67%
8.17%
10.35%
10.56%
12.41%, 3.00% PIK
10.55%
10.67%
11.03%
9.38%
11.67%
8.88%
11.42%
11.49%
10.19%
9.45%
1.50%
10.17%
12.42%
10.45%
9.42%
12.00%
12.70%
12.72%
12.20% PIK
0.75%
11.50% PIK
11.59%
12.63% PIK
12.64% PIK
8.62%
9.17%
15.00% PIK
10.82%
8.44%
7.25%
7.28%
10.92%
9.16%
10.42%
8.95%
8.96%
11.19%, 2.0% PIK
11.51%
9.95%
8.68%
8.87%
8.92%
9.20%
8.20%
12.38% PIK
11.95%
8.15%
10.09%
9.12%
12.71%
16.18%
9.41%
10.76% PIK
9.96%
13.00%
10.65%
14.25% PIK
12.25%
9.67%
9.77%
10.49%, 3.00% PIK
8.66%
8.90%
10.20%
11.27%
11.45%, 6.00% PIK
13.79%
11.65%
11.67%, 4.50% PIK
10.71%
10.27%
10.41%
14.00% PIK
7.17%
11.17%, 3.50% PIK
16.00% PIK
10.07%
7.92%
